# Data Loading

In [1]:
import pandas as pd
import numpy as np 
import json

# load the raw JSON data
with open('raw_credit_applications.json') as file:
    raw_data = json.load(file)

# flatten the standard nested objects and display the new column names
df = pd.json_normalize(raw_data)
print(df.columns)

Index(['_id', 'spending_behavior', 'processing_timestamp',
       'applicant_info.full_name', 'applicant_info.email',
       'applicant_info.ssn', 'applicant_info.ip_address',
       'applicant_info.gender', 'applicant_info.date_of_birth',
       'applicant_info.zip_code', 'financials.annual_income',
       'financials.credit_history_months', 'financials.debt_to_income',
       'financials.savings_balance', 'decision.loan_approved',
       'decision.rejection_reason', 'loan_purpose', 'decision.interest_rate',
       'decision.approved_amount', 'financials.annual_salary', 'notes'],
      dtype='object')


# Data Handling

In [2]:
# handle the spending_behavior array of dictionaries 
spending_exploded = df[['_id', 'spending_behavior']].explode('spending_behavior').dropna(subset=['spending_behavior'])

# flatten these new exploded column since it includes category and amount
spending_normalize = pd.json_normalize(spending_exploded['spending_behavior'])
spending_normalize.insert(0, '_id', spending_exploded['_id'].values)

# pivot the data to have one row per id with spending categories as columns
spending_pivot = spending_normalize.pivot_table(
    index='_id',
    columns='category',
    values='amount',
    fill_value='-'
).add_prefix('spending_')

# join these new spending columns with the main df and drop the old raw array
df_clean = df.drop(columns=['spending_behavior'])
df_clean = df_clean.merge(spending_pivot, on='_id', how='left')

display(df_clean.head())

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,...,spending_Fitness,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,...,-,-,-,-,-,790.0,480.0,-,-,-
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,...,-,-,-,243.0,-,608.0,-,-,-,-
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,...,-,-,-,-,-,109.0,-,-,-,-
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,...,575.0,-,-,-,-,-,-,-,-,-
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,...,-,-,-,-,-,-,-,-,-,-


# Intentional Data Issues Assessment

### 1. Duplicate Records Assessment

In [3]:
# find exact duplicates and count them
duplicates = df_clean[df_clean.duplicated(subset=['_id'], keep=False)]
duplicate_count = df_clean.duplicated(subset=['_id']).sum()
duplicate_percentage = (duplicate_count / len(df_clean)) * 100

# display the duplicates 
print(f'Found {duplicate_count} duplicated applications, which represent {duplicate_percentage:.2f}% of the total:')
display(duplicates[['_id', 'applicant_info.full_name', 'applicant_info.email']])

Found 2 duplicated applications, which represent 0.40% of the total:


,_id,applicant_info.full_name,applicant_info.email
8,app_042,Joseph Lopez,joseph.lopez1@gmail.com
354,app_042,Joseph Lopez,joseph.lopez1@gmail.com
383,app_001,Stephanie Nguyen,stephanie.nguyen47@mail.com
455,app_001,Stephanie Nguyen,stephanie.nguyen47@mail.com


### 2. Missing and Incomplete Records Assessment

In [4]:
# take care of empty strings by changing them to NaN values
df_clean.replace('', np.nan, inplace=True)

# count total missing values per column
missing_data = df_clean.isnull().sum()
missing_data = missing_data[missing_data > 0]

for column, count in missing_data.items():
    percentage = (count / len(df_clean)) * 100
    print(f"'{column}': {count} missing values ({percentage:.2f}%)")

'processing_timestamp': 440 missing values (87.65%)
'applicant_info.email': 7 missing values (1.39%)
'applicant_info.ssn': 5 missing values (1.00%)
'applicant_info.ip_address': 5 missing values (1.00%)
'applicant_info.gender': 3 missing values (0.60%)
'applicant_info.date_of_birth': 5 missing values (1.00%)
'applicant_info.zip_code': 2 missing values (0.40%)
'financials.annual_income': 5 missing values (1.00%)
'decision.rejection_reason': 292 missing values (58.17%)
'loan_purpose': 452 missing values (90.04%)
'decision.interest_rate': 210 missing values (41.83%)
'decision.approved_amount': 210 missing values (41.83%)
'financials.annual_salary': 497 missing values (99.00%)
'notes': 500 missing values (99.60%)


### 3. Inconsistent Data Types Assessment